# Module 1: Building a Single Agent with Role-Based Access Control

## Overview

In this module, you'll build a **single Product Catalog Agent** with role-based access control (RBAC) using the Strands Agent SDK and **MCP (Model Context Protocol)** tools.

### Why Start with a Single Agent?
Before building complex multi-agent systems, it's critical to:
1. **Validate tool calling** - Ensure the agent correctly selects and uses the right tools
2. **Test access control** - Verify that role-based restrictions work properly
3. **Evaluate effectiveness** - Measure a single agent's quality before adding orchestration complexity

### What You'll Build

```
┌──────────────────────────────────┐
│      User Request               │
│  (with role: customer or admin)  │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│   PRODUCT CATALOG AGENT          │
│   (Claude Haiku 4.5)             │
│   • Role-aware system prompt     │
│   • Tool filtering by role       │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│   Product MCP Server (FastMCP)   │
│                                  │
│  READ tools (customer + admin):  │
│  • search_products               │
│  • get_product_details           │
│  • check_inventory               │
│  • get_product_recommendations   │
│  • compare_products              │
│  • get_return_policy             │
│                                  │
│  WRITE tools (admin only):       │
│  • create_product                │
│  • update_product                │
│  • delete_product                │
│  • update_inventory              │
│  • update_pricing                │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│       DynamoDB Products Table    │
└──────────────────────────────────┘
```

### Learning Objectives
1. Build an agent with Strands SDK connected to an MCP tool server
2. Understand MCP architecture: agent ↔ MCP client ↔ MCP server ↔ DynamoDB
3. Implement role-based access control by filtering tools per user role
4. Test different personas (customer vs admin) against the same agent
5. Validate access control boundaries

## Step 1: Environment Setup

First, let's set up our environment and verify we have access to the required resources.

In [ ]:
# Install dependencies if needed
!pip install -q strands-agents strands-agents-tools boto3 mcp

In [ ]:
import boto3
import json
import os
import sys
from datetime import datetime

# Get AWS region
session = boto3.Session()
REGION = session.region_name or 'us-west-2'
print(f"AWS Region: {REGION}")

# Set environment variables for tools
os.environ['AWS_REGION'] = REGION

# Get table names from SSM (set by workshop infrastructure)
ssm = boto3.client('ssm', region_name=REGION)

try:
    os.environ['PRODUCTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-products-table')['Parameter']['Value']
    print(f"Products Table: {os.environ['PRODUCTS_TABLE_NAME']}")
except Exception as e:
    print(f"Note: Could not retrieve SSM parameters ({e})")
    print("Using default table names - ensure infrastructure is deployed")
    os.environ['PRODUCTS_TABLE_NAME'] = 'ecommerce-workshop-products'

## Step 2: Understanding MCP Tools

Our agent uses **MCP (Model Context Protocol)** to access tools from a separate MCP server process. This provides:
- **Decoupling**: Tools run as a separate process from the agent
- **Reusability**: MCP servers can be shared across multiple agents
- **Standardization**: Follows the MCP protocol for tool discovery and invocation

### MCP Server Architecture

```
Agent (Haiku)  ──MCPClient──►  MCP Server  ──►  DynamoDB
                  (stdio)      (FastMCP)        Products Table
```

### Tools in Our MCP Server

| Tool | Access Level | Description |
|------|-------------|-------------|
| `search_products` | Customer + Admin | Search catalog by keywords/category |
| `get_product_details` | Customer + Admin | Get full product details |
| `check_inventory` | Customer + Admin | Check stock availability |
| `get_product_recommendations` | Customer + Admin | Get product recommendations |
| `compare_products` | Customer + Admin | Side-by-side product comparison |
| `get_return_policy` | Customer + Admin | Return policy information |
| `create_product` | **Admin only** | Add new product to catalog |
| `update_product` | **Admin only** | Modify product information |
| `delete_product` | **Admin only** | Remove product (soft delete) |
| `update_inventory` | **Admin only** | Adjust stock quantities |
| `update_pricing` | **Admin only** | Change prices, set sales |

In [ ]:
# Verify MCP server exists
from pathlib import Path

mcp_server_path = Path('mcp_servers/product_mcp_server.py')
print(f"Product MCP Server: {mcp_server_path} - {'exists' if mcp_server_path.exists() else 'MISSING'}")

## Step 3: Build the Product Catalog Agent (No RBAC)

Let's first build a basic agent that connects to the MCP server with **all tools available** (no role filtering yet). This lets us verify the MCP connection and tool calling work correctly.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

# Model configuration
HAIKU_MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
print(f"Model: {HAIKU_MODEL_ID}")

# Connect to the product MCP server
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(mcp_server_path)],
    env={
        **os.environ,
        "AWS_REGION": REGION,
        "PRODUCTS_TABLE": os.environ.get('PRODUCTS_TABLE_NAME', 'ecommerce-workshop-products')
    }
)

mcp_client = MCPClient(lambda: stdio_client(server_params))
mcp_client.__enter__()

# Discover all tools from the MCP server
all_tools = mcp_client.list_tools_sync()
print(f"\nDiscovered {len(all_tools)} tools from MCP server:")
for tool in all_tools:
    print(f"  - {tool.tool_name}")

In [ ]:
# Create a basic agent with ALL tools (no RBAC yet)
basic_agent = Agent(
    name="ProductCatalogAgent",
    model=BedrockModel(
        model_id=HAIKU_MODEL_ID,
        region_name=REGION,
        temperature=0.3,
        max_tokens=1500
    ),
    system_prompt="You are a Product Catalog Assistant. Help users with product queries using the available tools.",
    tools=all_tools
)
print("Basic agent created with all 11 tools (no role filtering)")

## Step 4: Test Basic Tool Calling

Let's verify the agent correctly uses MCP tools to answer product queries.

In [ ]:
# Test 1: Product search
print("=" * 60)
print("Test 1: Product Search")
print("=" * 60)
response = basic_agent("Do you have any wireless headphones under $100?")
print(f"\nResponse: {response}")

In [ ]:
# Test 2: Product details
print("=" * 60)
print("Test 2: Product Details")
print("=" * 60)
response = basic_agent("Tell me about the Gaming Mechanical Keyboard PROD-077")
print(f"\nResponse: {response}")

In [ ]:
# Test 3: Inventory check
print("=" * 60)
print("Test 3: Inventory Check")
print("=" * 60)
response = basic_agent("Is the 4K webcam PROD-088 in stock?")
print(f"\nResponse: {response}")

In [ ]:
# Test 4: Product comparison
print("=" * 60)
print("Test 4: Product Comparison")
print("=" * 60)
response = basic_agent("Compare the Wireless Bluetooth Headphones PROD-001 with the Noise Canceling Earbuds PROD-055")
print(f"\nResponse: {response}")

In [ ]:
# Clean up the basic agent's MCP connection
mcp_client.__exit__(None, None, None)
print("Basic agent MCP connection cleaned up")

## Step 5: Understanding Role-Based Access Control (RBAC)

Now that we've confirmed the agent and tools work, let's add **access control**.

### The Problem
Our MCP server exposes 11 tools - 6 read-only tools (safe for customers) and 5 write tools (admin-only). Without RBAC, any user could create, update, or delete products.

### The Solution: Tool Filtering

We implement RBAC by **filtering which tools the agent receives** based on the user's role:

```python
# Customer gets 6 read-only tools
customer_tools = filter_by_role(all_tools, role="customer")  # 6 tools

# Admin gets all 11 tools
admin_tools = filter_by_role(all_tools, role="admin")  # 11 tools
```

Since the LLM can only call tools it knows about, **removing a tool from the agent's tool list is an effective access control boundary**.

### Production Mapping (Module 03)
In production, user roles come from **AgentCore Identity** via JWT tokens:
- A **Cognito User Pool** manages users with `customer` and `admin` groups
- The **JWT inbound authorizer** on AgentCore Runtime validates tokens
- Agent code extracts the role from JWT claims and filters tools accordingly

In [ ]:
# Add agents to path and import the RBAC-enabled agent
sys.path.insert(0, 'agents')
from product_catalog_agent import (
    ProductCatalogAgent,
    UserSession,
    create_product_catalog_agent,
    CUSTOMER_TOOLS,
    ADMIN_TOOLS,
    ADMIN_ONLY_TOOLS,
    CUSTOMER_PERSONAS,
    ADMIN_PERSONAS,
)

print("RBAC Configuration:")
print(f"  Customer tools ({len(CUSTOMER_TOOLS)}): {CUSTOMER_TOOLS}")
print(f"  Admin-only tools ({len(ADMIN_ONLY_TOOLS)}): {ADMIN_ONLY_TOOLS}")
print(f"  Total admin tools ({len(ADMIN_TOOLS)}): {len(ADMIN_TOOLS)}")

## Step 6: Create User Sessions (Personas)

Let's define our test personas. In production, these would come from JWT claims.

In [ ]:
# Define test personas
customer_session = UserSession(
    user_id="CUST-1001",
    role="customer",
    email="john.smith@email.com",
    name="John Smith"
)

admin_session = UserSession(
    user_id="ADMIN-001",
    role="admin",
    email="alice.admin@company.com",
    name="Alice (Admin)"
)

print("Test Personas:")
print(f"  Customer: {customer_session.name} ({customer_session.email}) - role: {customer_session.role}")
print(f"  Admin:    {admin_session.name} ({admin_session.email}) - role: {admin_session.role}")

## Step 7: Test Customer Persona

The customer agent should be able to search, view, and compare products, but should **NOT** be able to create, update, or delete products.

In [ ]:
# Create agent with CUSTOMER role
customer_agent = create_product_catalog_agent(
    region=REGION,
    user_session=customer_session
)

user_info = customer_agent.get_user_info()
print(f"Agent created for: {user_info['name']} (role: {user_info['role']})")
print(f"Available tools ({user_info['tools_available']}): {user_info['tools']}")

In [ ]:
# Customer Test 1: Search (should work)
print("=" * 60)
print("Customer Test 1: Product Search (ALLOWED)")
print("=" * 60)
response = customer_agent("I'm looking for a good gaming keyboard with RGB lighting")
print(f"\nResponse: {response}")

In [ ]:
# Customer Test 2: Recommendations (should work)
print("=" * 60)
print("Customer Test 2: Recommendations (ALLOWED)")
print("=" * 60)
response = customer_agent("Can you recommend some audio products for a music lover?")
print(f"\nResponse: {response}")

In [ ]:
# Customer Test 3: Attempt admin action (should be REFUSED)
print("=" * 60)
print("Customer Test 3: Create Product (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent("Create a new product called 'Super Speaker' with price $299.99 in the Audio category")
print(f"\nResponse: {response}")

In [ ]:
# Customer Test 4: Attempt to delete a product (should be REFUSED)
print("=" * 60)
print("Customer Test 4: Delete Product (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent("Delete the product PROD-001 from the catalog")
print(f"\nResponse: {response}")

In [ ]:
# Clean up customer agent
customer_agent.cleanup()
print("Customer agent cleaned up")

## Step 8: Test Admin Persona

The admin agent should have **full access** to all tools, including creating, updating, and deleting products.

In [ ]:
# Create agent with ADMIN role
admin_agent = create_product_catalog_agent(
    region=REGION,
    user_session=admin_session
)

user_info = admin_agent.get_user_info()
print(f"Agent created for: {user_info['name']} (role: {user_info['role']})")
print(f"Available tools ({user_info['tools_available']}): {user_info['tools']}")

In [ ]:
# Admin Test 1: Search still works (admin has all customer tools too)
print("=" * 60)
print("Admin Test 1: Product Search (ALLOWED)")
print("=" * 60)
response = admin_agent("Search for all audio products")
print(f"\nResponse: {response}")

In [ ]:
# Admin Test 2: Create a new product
print("=" * 60)
print("Admin Test 2: Create Product (ALLOWED)")
print("=" * 60)
response = admin_agent(
    "Create a new product with ID PROD-200 called 'Premium Gaming Headset' "
    "in the Audio category for $129.99. Description: 'Professional gaming headset "
    "with 7.1 surround sound, detachable microphone, and RGB lighting.' "
    "Specifications: bluetooth 5.3, weight 320g, driver size 50mm. "
    "Set initial stock to 50 units."
)
print(f"\nResponse: {response}")

In [ ]:
# Admin Test 3: Update pricing with a sale
print("=" * 60)
print("Admin Test 3: Update Pricing (ALLOWED)")
print("=" * 60)
response = admin_agent(
    "Set a sale price of $99.99 for PROD-200 (regular price stays $129.99) "
    "until 2025-03-31"
)
print(f"\nResponse: {response}")

In [ ]:
# Admin Test 4: Update inventory
print("=" * 60)
print("Admin Test 4: Update Inventory (ALLOWED)")
print("=" * 60)
response = admin_agent("Set the inventory for PROD-088 (4K webcam) to 100 units")
print(f"\nResponse: {response}")

In [ ]:
# Admin Test 5: Delete (discontinue) a product
print("=" * 60)
print("Admin Test 5: Delete Product (ALLOWED - soft delete)")
print("=" * 60)
response = admin_agent("Discontinue product PROD-200")
print(f"\nResponse: {response}")

In [ ]:
# Clean up admin agent
admin_agent.cleanup()
print("Admin agent cleaned up")

## Step 9: Access Control Boundary Test

Let's run a cross-role validation to confirm the RBAC boundary holds:
1. Admin creates a product → Customer can find it via search
2. Customer cannot modify the product the admin created

In [ ]:
# Create both agents
admin_agent = create_product_catalog_agent(region=REGION, user_session=admin_session)
customer_agent = create_product_catalog_agent(region=REGION, user_session=customer_session)

print(f"Admin tools:    {admin_agent.get_user_info()['tools_available']} tools")
print(f"Customer tools: {customer_agent.get_user_info()['tools_available']} tools")

In [ ]:
# Step A: Admin creates a new product
print("=" * 60)
print("Step A: Admin creates product PROD-300")
print("=" * 60)
response = admin_agent(
    "Create product PROD-300 called 'Wireless Charging Pad' in Accessories category "
    "for $39.99. Description: 'Fast wireless charger compatible with all Qi devices, "
    "supports up to 15W charging.' Specs: charging speed 15W, compatibility Qi, weight 120g. "
    "Stock: 200 units."
)
print(f"\nAdmin Response: {response}")

In [ ]:
# Step B: Customer can find the new product
print("=" * 60)
print("Step B: Customer searches for the new product")
print("=" * 60)
response = customer_agent("Do you have any wireless charging pads?")
print(f"\nCustomer Response: {response}")

In [ ]:
# Step C: Customer tries to modify the product (SHOULD FAIL)
print("=" * 60)
print("Step C: Customer tries to change the price (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent("Change the price of PROD-300 to $19.99")
print(f"\nCustomer Response: {response}")

In [ ]:
# Step D: Clean up - admin deletes the test product
print("=" * 60)
print("Step D: Admin cleans up test product")
print("=" * 60)
response = admin_agent("Delete product PROD-300")
print(f"\nAdmin Response: {response}")

In [ ]:
# Clean up both agents
admin_agent.cleanup()
customer_agent.cleanup()
print("Both agents cleaned up")

## Step 10: From Prototype to Production — AgentCore Identity

In this module, we simulated RBAC using a local `UserSession` class. In **Module 3** (Production Deployment), this maps to real authentication using **Amazon Bedrock AgentCore Identity**:

### Production Architecture

```
┌────────────┐     JWT Token      ┌─────────────────────┐
│  Web App   │ ──────────────────► │  AgentCore Runtime  │
│  (React)   │                     │                     │
└─────┬──────┘                     │  JWT Authorizer     │
      │                            │  validates token    │
      │ Login                      │                     │
      ▼                            │  Agent code reads   │
┌─────────────┐                    │  role from claims   │
│  Cognito    │                    │                     │
│  User Pool  │                    │  Tools filtered     │
│             │                    │  by role            │
│ Groups:     │                    └─────────────────────┘
│ - customer  │
│ - admin     │
└─────────────┘
```

### Key Mapping

| Prototype (Module 01) | Production (Module 03) |
|----------------------|------------------------|
| `UserSession` class | JWT token claims |
| `role` field | Cognito group membership |
| `user_id` field | JWT `sub` claim |
| `email` field | JWT `email` claim |
| Local tool filtering | Same logic, but role from verified JWT |

### Why This Pattern Works
1. **Defense in depth**: Tool filtering prevents the LLM from even knowing about restricted tools
2. **System prompt reinforcement**: Role-aware prompts add a second layer of guidance
3. **Verifiable**: You can inspect which tools an agent received at construction time
4. **Standard OAuth/JWT**: AgentCore Identity uses industry-standard authentication

## Summary

In this module, you built a **single Product Catalog Agent** with role-based access control that:

1. **Connects to an MCP server** with 11 tools (6 read + 5 write)
2. **Filters tools by role** — customers get 6 read tools, admins get all 11
3. **Uses role-aware system prompts** — different instructions per persona
4. **Validates access control** — customer cannot perform admin operations

### Key Takeaways
- **Start simple**: Validate one agent thoroughly before building multi-agent systems
- **RBAC via tool filtering**: Removing tools from the agent is a strong access control boundary
- **Persona testing**: Same agent code, different behaviors based on role
- **Production-ready pattern**: Local prototype maps directly to AgentCore Identity JWT auth

### Key Files

| File | Description |
|------|-------------|
| `mcp_servers/product_mcp_server.py` | MCP server with 11 product tools (6 read + 5 admin) |
| `agents/product_catalog_agent.py` | Agent with RBAC (UserSession, tool filtering, role prompts) |

### Next Steps

In **Module 2**, we'll evaluate this agent with a comprehensive test dataset to establish baseline metrics for quality, tool accuracy, and access control compliance.

In [ ]:
# Save region for use in next module
%store REGION
print("Session data saved for Module 2!")